# Version 3: factorized inference submission

This solution uses only B -> A -> C contraction order, a fused custom CUDA kernel,
and one-time factor prepacking. The benchmark, tests, reference, package exports,
and dependencies remain unchanged. The stock harness calls prepare_optimized
and prints the existing factorized_optimized rows.

Run from the repository root using Python 3.11 or newer. GPU execution requires
CUDA-enabled PyTorch, nvcc, ninja, a C++ compiler, and sm_75 or later.
For notebook execution install jupyterlab and ipykernel in the same environment.

GPU speedup is not established by the implementation. The previous screenshots
showed an unfavorable rank-8/token-1 result; this kernel must be measured again.
All GPU result cells explicitly skip when CUDA is unavailable.

Saved local outputs were produced with a Python cell runner because Jupyter
is not installed on this machine: 24 FP64 cases passed; the stock suite
passed 15 tests and skipped 6 CUDA checks. Full Jupyter execution, notebook
rendering, CUDA compilation/correctness, and GPU measurements remain pending.

The implementation is AI-assisted original code. The CPU algebra and interface
checks are independently executable below; GPU verification is a separate step.


## Environment

The notebook generates the package files imported by isolated benchmark/test subprocesses.


In [1]:
from pathlib import Path
import contextlib
import importlib
import json
import math
import subprocess
import sys
from unittest import mock

import torch

ROOT = Path.cwd()
assert (ROOT / "benchmarks/benchmark.py").exists(), "Run from the repository root"
sys.path.insert(0, str(ROOT / "src"))
torch.set_num_threads(1)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
HAS_CUDA = torch.cuda.is_available()
print({"python": sys.version.split()[0], "executable": sys.executable,
       "torch": str(torch.__version__), "cuda_runtime": torch.version.cuda,
       "cuda_available": HAS_CUDA})
if HAS_CUDA:
    print({"gpu": torch.cuda.get_device_name(),
           "capability": torch.cuda.get_device_capability()})
    subprocess.run(["nvidia-smi"], check=True)


{'python': '3.14.5', 'executable': '/Users/olorin/ws/251004_alexk_projects/files/260926_Ishwar_GpuOpt/.venv/bin/python', 'torch': '2.14.0', 'cuda_runtime': None, 'cuda_available': False}


## Algebra and Layout

Starting from the tensor-ring definition:

    U[t,k,q,i,b,c] = sum_j x[t,i,j,k] B[b,q,j,c]
    V[t,k,q,a,p,c] = sum_i,b A[a,p,i,b] U[t,k,q,i,b,c]
    y[t,p,q,r]    = sum_k,a,c V[t,k,q,a,p,c] C[c,r,k,a]
                 = sum_i,j,k,a,b,c x[t,i,j,k] A[a,p,i,b] B[b,q,j,c] C[c,r,k,a]

Only finite sums are reordered, so the real-valued result is identical.
Floating-point rounding changes; the assignment tolerances are preserved.

Packing snapshots A[a,p,i,b], Bp[j,b,q,c], and Cp[k,a,c,r] once.
There is no weight contraction during preparation, no dense W in inference,
and no module-level weight cache.

Each CUDA block owns a full (token,q) slice. For each k, it computes U, computes
all V[a,p,c] in parallel, and accumulates y[p,r] over all a,c. Two block barriers
per k protect the disjoint U/V storage. Each output has one writer.
No k/a/c reduction is partitioned across blocks and no partial output is stored.

U/V retain the operand dtype at contraction boundaries. CUDA FP16 dot reductions
use ordinary FP32 arithmetic for correctness, as in PyTorch's reference behavior;
the solution introduces no separate mixed-precision optimization.
There are no Tensor Core specializations, split-K, atomics, autotuning,
torch.compile, or CUDA Graphs.

CUDA C++ is permitted by README.md. It preserves T4 support without a Triton
dependency; current [Triton compatibility guidance](https://github.com/triton-lang/triton#compatibility)
lists NVIDIA compute capability 8.0+.


In [2]:
for rank in (8, 16):
    i, j, k = 8, 12, 20
    p, q, s = 12, 10, 24
    abc = i*j*k*p*rank**2 + j*k*p*q*rank**3 + k*p*q*s*rank**2
    bac = i*j*k*q*rank**2 + i*k*p*q*rank**3 + k*p*q*s*rank**2
    shared_fp16 = (i*rank**2 + rank*p*rank)*2 + p*s*4
    print({"rank": rank, "analytic_abc_multiply_adds_per_token": abc,
           "analytic_bac_multiply_adds_per_token": bac,
           "analytic_reduction_percent": round(100*(1-bac/abc), 2),
           "shared_bytes_per_fp16_block": shared_fp16})
print("Analytic savings are not measured speedups; token 1 launches only 10 blocks.")


{'rank': 8, 'analytic_abc_multiply_adds_per_token': 19906560, 'analytic_bac_multiply_adds_per_token': 14745600, 'analytic_reduction_percent': 25.93, 'shared_bytes_per_fp16_block': 3712}
{'rank': 16, 'analytic_abc_multiply_adds_per_token': 138608640, 'analytic_bac_multiply_adds_per_token': 98304000, 'analytic_reduction_percent': 29.08, 'shared_bytes_per_fp16_block': 11392}
Analytic savings are not measured speedups; token 1 launches only 10 blocks.


## CUDA Kernel Source

This cell is the source of truth for the generated kernel module.


In [3]:
%%writefile src/factorized_inference/tr_kernels.py
# GENERATED by solution.ipynb - edit the notebook, not this file
"""Version 3 fused B -> A -> C execution using original factorized cores.

Each block owns one (token, q) and completes the entire k/a/c reduction.
U[i,b,c] and V[a,p,c] stay in shared memory. There is a unique output
writer, no global intermediate, no partial-output buffer, and no atomics.
The CUDA C++ code is original AI-assisted code, not adapted library code.
"""

from __future__ import annotations

import hashlib
import os
import threading

import torch


CPP_SOURCE = r"""
#include <torch/extension.h>
#include <vector>
torch::Tensor version3_cuda(torch::Tensor x, torch::Tensor a,
    torch::Tensor bp, torch::Tensor cp, std::vector<int64_t> dims);
"""

CUDA_SOURCE = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAException.h>
#include <cuda.h>
#include <cuda_runtime.h>
#include <limits>
#include <vector>

template<typename T>
__global__ void version3_bac_fused(
    const T* __restrict__ x, const T* __restrict__ a,
    const T* __restrict__ bp, const T* __restrict__ cp,
    T* __restrict__ out, int I, int J, int K, int P, int Q, int S, int R)
{
    const int q = blockIdx.x % Q;
    const int64_t t = blockIdx.x / Q;
    const int nu = I*R*R, nv = R*P*R, ny = P*S;
    extern __shared__ __align__(16) unsigned char storage[];
    float* y = reinterpret_cast<float*>(storage);
    T* u = reinterpret_cast<T*>(y + ny);
    T* v = u + nu;
    for (int z=threadIdx.x; z<ny; z+=blockDim.x) y[z]=0.0f;

    for (int k=0; k<K; ++k) {
        // U[i,b,c] = sum_j x[t,i,j,k] * Bp[j,b,q,c].
        for (int z=threadIdx.x; z<nu; z+=blockDim.x) {
            const int c=z%R, b=(z/R)%R, i=z/(R*R);
            float acc=0.0f;
            for (int j=0; j<J; ++j)
                acc += float(x[t*(int64_t(I)*J*K)+(int64_t(i)*J+j)*K+k])
                    * float(bp[((int64_t(j)*R+b)*Q+q)*R+c]);
            u[z]=T(acc);
        }
        // Also completes all V readers from the preceding k before reuse.
        __syncthreads();

        // All a/p/c values are independent; threads fill the full V tile.
        for (int z=threadIdx.x; z<nv; z+=blockDim.x) {
            const int c=z%R, p=(z/R)%P, aa=z/(P*R);
            float acc=0.0f;
            for (int ib=0; ib<I*R; ++ib)
                acc += float(a[(int64_t(aa)*P+p)*I*R+ib])
                    * float(u[ib*R+c]);
            v[z]=T(acc);
        }
        __syncthreads();

        // A thread owns y[p,r] for all k; no cross-block reduction is needed.
        for (int z=threadIdx.x; z<ny; z+=blockDim.x) {
            const int p=z/S, r=z%S;
            float acc=y[z];
            for (int aa=0; aa<R; ++aa)
                for (int c=0; c<R; ++c)
                    acc += float(v[(aa*P+p)*R+c])
                        * float(cp[((int64_t(k)*R+aa)*R+c)*S+r]);
            y[z]=acc;
        }
        // U and V occupy disjoint storage. Next k may refill U while slower
        // threads still read V; the first barrier prevents early V overwrite.
    }
    for (int z=threadIdx.x; z<ny; z+=blockDim.x) {
        const int p=z/S, r=z%S;
        out[((t*P+p)*Q+q)*S+r]=T(y[z]);
    }
}

static void check_tensor(const torch::Tensor& value, const torch::Tensor& x) {
    TORCH_CHECK(value.is_cuda() && value.device()==x.device(), "CUDA device mismatch");
    TORCH_CHECK(value.scalar_type()==x.scalar_type(), "dtype mismatch");
    TORCH_CHECK(value.is_contiguous(), "CUDA tensors must be contiguous");
}

torch::Tensor version3_cuda(torch::Tensor x, torch::Tensor a,
    torch::Tensor bp, torch::Tensor cp, std::vector<int64_t> dims)
{
    TORCH_CHECK(dims.size()==7, "Expected I,J,K,P,Q,S,R");
    for (auto d : dims) TORCH_CHECK(d>0 && d<=4096, "Unsupported dimension");
    const int I=dims[0], J=dims[1], K=dims[2], P=dims[3];
    const int Q=dims[4], S=dims[5], R=dims[6];
    TORCH_CHECK(x.is_cuda() && x.dim()==2 && x.size(1)==int64_t(I)*J*K,
        "Incorrect CUDA input shape");
    TORCH_CHECK(x.scalar_type()==at::kHalf || x.scalar_type()==at::kFloat,
        "Version 3 CUDA requires FP16/FP32");
    check_tensor(x,x); check_tensor(a,x); check_tensor(bp,x); check_tensor(cp,x);
    TORCH_CHECK(a.sizes()==at::IntArrayRef({R,P,I,R}), "A shape mismatch");
    TORCH_CHECK(bp.sizes()==at::IntArrayRef({J,R,Q,R}), "Bp shape mismatch");
    TORCH_CHECK(cp.sizes()==at::IntArrayRef({K,R,R,S}), "Cp shape mismatch");
    c10::cuda::CUDAGuard guard(x.device());
    const int64_t tokens=x.size(0), blocks=tokens*Q;
    TORCH_CHECK(blocks<=std::numeric_limits<int>::max(), "Too many blocks");
    const int64_t shared_bytes=(int64_t(I)*R*R+int64_t(R)*P*R)*x.element_size()
        + int64_t(P)*S*sizeof(float);
    // Use the ordinary shared-memory budget; no opt-in launch configuration.
    const auto* properties=at::cuda::getDeviceProperties(x.get_device());
    TORCH_CHECK(shared_bytes<=properties->sharedMemPerBlock,
        "Version 3 tile exceeds per-block shared memory");
    auto out=torch::empty({tokens,int64_t(P)*Q*S}, x.options());
    if (!tokens) return out;
    const auto stream=c10::cuda::getCurrentCUDAStream(x.get_device()).stream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "version3_bac", [&] {
        version3_bac_fused<scalar_t><<<blocks,256,shared_bytes,stream>>>(
            x.data_ptr<scalar_t>(), a.data_ptr<scalar_t>(), bp.data_ptr<scalar_t>(),
            cp.data_ptr<scalar_t>(), out.data_ptr<scalar_t>(), I,J,K,P,Q,S,R);
    });
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return out;
}
"""

_extension = None
_extension_lock = threading.Lock()


def _get_extension():
    """Cache compiled code only, never factor values or inferred dense weights."""
    global _extension
    if _extension is not None:
        return _extension
    with _extension_lock:
        if _extension is None:
            from torch.utils.cpp_extension import CUDA_HOME, load_inline

            if CUDA_HOME is None:
                raise RuntimeError("Version 3 requires nvcc/CUDA toolkit and a C++ compiler")
            digest = hashlib.sha256((CPP_SOURCE + CUDA_SOURCE).encode()).hexdigest()[:12]
            _extension = load_inline(
                name=f"tr_version3_bac_{digest}",
                cpp_sources=CPP_SOURCE,
                cuda_sources=CUDA_SOURCE,
                functions=["version3_cuda"],
                extra_cflags=["-O3"],
                extra_cuda_cflags=[
                    "-O3",
                    "-gencode=arch=compute_75,code=sm_75",
                    "-gencode=arch=compute_75,code=compute_75",
                ],
                with_cuda=True,
                verbose=os.environ.get("TR_BUILD_VERBOSE", "0") == "1",
            )
    return _extension


class PreparedCUDA:
    """Packed factor snapshots, fresh outputs, caller's current CUDA stream."""

    def __init__(self, a, bp, cp, spec):
        self.a, self.b, self.c = a, bp, cp
        self.device, self.dtype = a.device, a.dtype
        self.n_in = spec.in_features
        self.dims = list(spec.input_modes + spec.output_modes + (spec.rank,))
        if self.dtype not in (torch.float16, torch.float32):
            raise ValueError("Version 3 CUDA requires FP16/FP32 cores")
        if any(d > 4096 for d in self.dims):
            raise ValueError("Version 3 CUDA dimensions must be at most 4096")
        if torch.cuda.get_device_capability(self.device) < (7, 5):
            raise RuntimeError("Version 3 CUDA build requires sm_75 or later")
        i = spec.input_modes[0]
        p, _, s = spec.output_modes
        self.shared_bytes = (
            (i * spec.rank**2 + spec.rank * p * spec.rank) * a.element_size()
            + p * s * 4
        )
        limit = torch.cuda.get_device_properties(self.device).shared_memory_per_block
        if self.shared_bytes > limit:
            raise ValueError(
                f"Version 3 needs {self.shared_bytes} shared bytes; device limit is {limit}"
            )
        with torch.cuda.device(self.device):
            self.ext = _get_extension()

    def __call__(self, x):
        if x.ndim != 2 or x.shape[1] != self.n_in:
            raise ValueError(f"Expected x.shape == [tokens, {self.n_in}]")
        if x.device != self.device or x.dtype != self.dtype:
            raise ValueError("Input device/dtype must match prepared cores")
        if x.requires_grad:
            raise ValueError("Version 3 is inference-only; detach the input")
        return self.ext.version3_cuda(x.contiguous(), self.a, self.b, self.c, self.dims)


Generated src/factorized_inference/tr_kernels.py


## Submission Integration

Use the existing submission entry points; no harness or export changes are needed.


In [4]:
%%writefile src/factorized_inference/submission.py
# GENERATED by solution.ipynb - edit the notebook, not this file
"""Submission API: Version 3 uses only B -> A -> C, fusion, and core packing."""

from __future__ import annotations

from typing import Callable, Sequence

import torch

from .reference import TRSpec


def _validate_cores(cores, spec):
    cores = tuple(cores)
    if len(cores) != 3:
        raise ValueError("Expected three cores")
    device, dtype = cores[0].device, cores[0].dtype
    if device.type not in ("cpu", "cuda"):
        raise ValueError("Version 3 supports CPU and CUDA")
    for index, (core, inputs, outputs) in enumerate(
        zip(cores, spec.input_modes, spec.output_modes)
    ):
        if tuple(core.shape) != (spec.rank, outputs, inputs, spec.rank):
            raise ValueError(f"Incorrect shape for core {index}: {tuple(core.shape)}")
        if core.device != device or core.dtype != dtype:
            raise ValueError("All cores must have the same device and dtype")
    allowed = ((torch.float16, torch.float32) if device.type == "cuda"
               else (torch.float16, torch.float32, torch.float64))
    if dtype not in allowed:
        raise ValueError("Unsupported core dtype for Version 3")
    return cores


def prepare_optimized(
    cores: Sequence[torch.Tensor],
    spec: TRSpec,
) -> Callable[[torch.Tensor], torch.Tensor]:
    """Snapshot/pack factors once; CUDA compilation also occurs here.

    A[a,p,i,b], Bp[j,b,q,c], Cp[k,a,c,r] retain the original factor dtype
    and element count. No weights are contracted or expanded in preparation.
    One callable accepts changed inputs/token counts and returns fresh outputs.

    Algebra: U=sum_j x*B; V=sum_i,b A*U; y=sum_k,a,c V*C.
    Substitution gives sum_i,j,k,a,b,c x*A*B*C, closing the same tensor ring.
    """
    a, b, c = _validate_cores(cores, spec)
    # clone() also snapshots cores that are already contiguous.
    a = a.detach().clone(memory_format=torch.contiguous_format)
    bp = b.detach().permute(2, 0, 1, 3).clone(memory_format=torch.contiguous_format)
    cp = c.detach().permute(2, 3, 0, 1).clone(memory_format=torch.contiguous_format)

    if a.is_cuda:
        from .tr_kernels import PreparedCUDA

        return PreparedCUDA(a, bp, cp, spec)

    def cpu_forward(x):
        if x.ndim != 2 or x.shape[1] != spec.in_features:
            raise ValueError(f"Expected x.shape == [tokens, {spec.in_features}]")
        if x.device != a.device or x.dtype != a.dtype:
            raise ValueError("Input device/dtype must match prepared cores")
        if x.requires_grad:
            raise ValueError("Version 3 is inference-only; detach the input")
        # CPU fallback verifies the reordered algebra. CUDA never enters here.
        x_modes = x.reshape(x.shape[0], *spec.input_modes)
        u = torch.einsum("tijk,jbqc->tkqibc", x_modes, bp)
        v = torch.einsum("tkqibc,apib->tkqapc", u, a)
        y = torch.einsum("tkqapc,kacr->tpqr", v, cp)
        return y.reshape(x.shape[0], spec.out_features)

    return cpu_forward


def tr_forward_optimized(
    x: torch.Tensor,
    cores: Sequence[torch.Tensor],
    spec: TRSpec,
) -> torch.Tensor:
    """One-shot inference; use prepare_optimized for repeated/timed calls."""
    return prepare_optimized(cores, spec)(x)


Generated src/factorized_inference/submission.py


In [5]:
from factorized_inference import TRSpec, make_cores, tr_forward_reference
from factorized_inference import dense_forward, materialize_dense_weight
import factorized_inference.tr_kernels as tr_kernels
import factorized_inference.submission as submission

importlib.reload(tr_kernels)
importlib.reload(submission)
# __init__.py may retain old bindings; call the reloaded submission module.
prepare = submission.prepare_optimized


## CPU Algebra and Interface Checks

The emulation follows the CUDA tile layout and k loop, independently of the
CPU fallback's pairwise contraction layout. Dense reconstruction below is only
a correctness oracle, never preparation or inference.


In [6]:
def oracle(x, cores, spec):
    weight = materialize_dense_weight(tuple(c.double() for c in cores), spec)
    return dense_forward(x.double(), weight)


def emulate_blocks(x, cores, spec):
    a, b, c = (core.double() for core in cores)
    i, j, k_count = spec.input_modes
    p, q_count, s = spec.output_modes
    rank = spec.rank
    xm = x.double().reshape(x.shape[0], i, j, k_count)
    bp = b.permute(2, 0, 1, 3)
    cp = c.permute(2, 3, 0, 1)
    result = torch.zeros(x.shape[0], p, q_count, s, dtype=torch.float64)
    for t in range(x.shape[0]):
        for q in range(q_count):
            for k in range(k_count):
                u = torch.einsum("ij,jbc->ibc", xm[t, :, :, k], bp[:, :, q, :])
                v = torch.einsum("apib,ibc->apc", a, u)
                result[t, :, q, :] += torch.einsum("apc,acr->pr", v, cp[k])
    return result.reshape(x.shape[0], spec.out_features)


specs = [TRSpec((1, 2, 3), (3, 1, 2), 1),
         TRSpec((2, 3, 2), (2, 4, 3), 3),
         TRSpec((4, 4, 4), (4, 4, 4), 8),
         TRSpec((4, 4, 4), (4, 4, 4), 16)]
checked = 0
for spec in specs:
    for seed in (0, 19):
        cores = make_cores(spec, dtype=torch.float64, seed=seed)
        snapshots = tuple(core.clone() for core in cores)
        run = prepare(cores, spec)
        for tokens, scale in ((1, 1.0), (3, 0.0), (2, 2.0)):
            generator = torch.Generator().manual_seed(seed + tokens + 37)
            x = torch.randn(tokens, spec.in_features, generator=generator,
                            dtype=torch.float64) * scale
            before = x.clone()
            expected = oracle(x, snapshots, spec)
            result = run(x)
            torch.testing.assert_close(result, expected, atol=1e-10, rtol=1e-10)
            torch.testing.assert_close(emulate_blocks(x, snapshots, spec), expected,
                                       atol=1e-10, rtol=1e-10)
            torch.testing.assert_close(result, tr_forward_reference(x, snapshots, spec),
                                       atol=1e-10, rtol=1e-10)
            torch.testing.assert_close(x, before, atol=0, rtol=0)
            for core, original in zip(cores, snapshots):
                torch.testing.assert_close(core, original, atol=0, rtol=0)
            saved = result.clone()
            run(x + 1)
            torch.testing.assert_close(result, saved, atol=0, rtol=0)
            checked += 1
        for core in cores:
            core.zero_()
        torch.testing.assert_close(run(x), expected, atol=1e-10, rtol=1e-10)
        assert run(x[:0]).shape == (0, spec.out_features)
print(f"CPU FP64 checks passed: {checked} cases, snapshots, empty inputs, fresh outputs.")


CPU FP64 checks passed: 24 cases, snapshots, empty inputs, fresh outputs.


## CUDA Correctness and Kernel Presence

The checks cover the five required workloads, changed inputs/token counts,
source-core mutation, two independent prepared operators, and a non-default
stream. High-level PyTorch contractions are forbidden while preparing/running
the optimized operator. A profile checks for a single custom GPU kernel.
On CPU-only hosts these checks are explicitly skipped, not counted as passing.


In [7]:
def check_result(actual, expected, dtype):
    tolerance = 0.02 if dtype == torch.float16 else 0.0001
    assert actual.shape == expected.shape
    assert actual.dtype == dtype and actual.device == expected.device
    assert torch.isfinite(actual).all()
    torch.testing.assert_close(actual.double(), expected, atol=tolerance, rtol=tolerance)


def errors(actual, expected):
    difference = (actual.double() - expected).abs()
    return {"max_absolute_error": difference.max().item(),
            "mean_absolute_error": difference.mean().item(),
            "relative_l2_error": (
                difference.norm() / expected.norm().clamp_min(1e-12)).item()}


def forbidden(*args, **kwargs):
    raise AssertionError("Optimized CUDA delegated its factorized contraction")


required_cases = [(8, 1), (8, 8), (8, 32), (16, 1), (16, 32)]
if not HAS_CUDA:
    print("SKIPPED: CUDA correctness, profiler, and stream checks require an NVIDIA GPU.")
else:
    for rank, tokens in required_cases:
        spec = TRSpec(rank=rank)
        for seed in (0, 19, 23):
            cores = make_cores(spec, device="cuda", dtype=torch.float16, seed=seed)
            snapshots = tuple(core.clone() for core in cores)
            generator = torch.Generator(device="cuda").manual_seed(seed + tokens + 37)
            x = torch.randn(tokens, spec.in_features, device="cuda",
                            dtype=torch.float16, generator=generator)
            before = x.clone()
            expected = oracle(x, snapshots, spec)
            reference = tr_forward_reference(x, snapshots, spec)
            with contextlib.ExitStack() as stack:
                for name in ("einsum", "matmul", "bmm", "mm", "compile"):
                    stack.enter_context(mock.patch.object(torch, name, forbidden))
                stack.enter_context(mock.patch.object(torch.nn.functional, "linear", forbidden))
                stack.enter_context(mock.patch(
                    "factorized_inference.reference.materialize_dense_weight", forbidden))
                run = prepare(cores, spec)
                result = run(x)
            check_result(result, expected, x.dtype)
            torch.testing.assert_close(result, reference, atol=0.02, rtol=0.02)
            torch.testing.assert_close(x, before, atol=0, rtol=0)
            for core, snapshot in zip(cores, snapshots):
                torch.testing.assert_close(core, snapshot, atol=0, rtol=0)
                core.zero_()
            check_result(run(x), expected, x.dtype)
            saved = result.clone()
            x2 = torch.randn(3, spec.in_features, device="cuda", dtype=x.dtype,
                             generator=generator)
            check_result(run(x2), oracle(x2, snapshots, spec), x.dtype)
            check_result(run(x2 * 0), torch.zeros_like(x2[:, :1]).double().expand(
                3, spec.out_features), x.dtype)
            torch.testing.assert_close(result, saved, atol=0, rtol=0)
            assert run(x[:0]).shape == (0, spec.out_features)
            stream = torch.cuda.Stream()
            stream.wait_stream(torch.cuda.current_stream())
            with torch.cuda.stream(stream):
                streamed = run(x)
            stream.synchronize()
            torch.testing.assert_close(streamed, result, atol=0, rtol=0)
            if seed == 0:
                torch.cuda.synchronize()
                activities = [torch.profiler.ProfilerActivity.CPU,
                              torch.profiler.ProfilerActivity.CUDA]
                with torch.profiler.profile(activities=activities) as prof:
                    run(x)
                    torch.cuda.synchronize()
                kernels = [event.name for event in prof.events()
                           if event.device_type == torch.autograd.DeviceType.CUDA]
                assert len(kernels) == 1 and "version3_bac_fused" in kernels[0], kernels
            print({"rank": rank, "tokens": tokens, "seed": seed,
                   **errors(result, expected)})
    small = TRSpec((2, 3, 2), (2, 4, 3), 3)
    for dtype in (torch.float16, torch.float32):
        first = make_cores(small, device="cuda", dtype=dtype, seed=1)
        second = make_cores(small, device="cuda", dtype=dtype, seed=2)
        run_first, run_second = prepare(first, small), prepare(second, small)
        x = torch.randn(4, small.in_features * 2, device="cuda", dtype=dtype)[:, ::2]
        for run, cores in ((run_first, first), (run_second, second), (run_first, first)):
            check_result(run(x), oracle(x, cores, small), dtype)
    print("CUDA correctness, independence, changing-input, kernel, and stream checks passed.")


SKIPPED: CUDA correctness, profiler, and stream checks require an NVIDIA GPU.


## Unmodified Test Suite

The supplied tests remain unchanged. CUDA skips mean GPU acceptance is still pending.


In [8]:
process = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True,
    env={**__import__("os").environ, "PYTHONPATH": str(ROOT / "src")},
)
print(process.stdout)
if process.stderr:
    print(process.stderr)
process.check_returncode()


..............ssssss.                                                    [100%]
=========================== short test summary info ============================
SKIPPED [1] tests/test_correctness.py:49: CUDA required for FP16 checks
SKIPPED [5] tests/test_correctness.py:68: CUDA required for final acceptance
15 passed, 6 skipped in 5.69s



## Required Benchmarks

These are the README commands with the original workloads, timing boundaries,
tolerances, method names, and output format. The dense baseline/oracle is
separate from the optimized factorized implementation. Compilation/packing
are included in preparation, not repeated in steady-state inference.

CPU-only runs do not create substitute GPU measurement files.


In [9]:
if not HAS_CUDA:
    print("SKIPPED: required CUDA benchmarks and GPU environment capture.")
else:
    subprocess.run([sys.executable, "benchmarks/benchmark.py", "--device", "cuda",
                    "--dtype", "float16", "--rank", "8", "--tokens", "1,8,32",
                    "--output", "results/rank8.json", "--profile-dir", "traces/rank8"],
                   check=True, cwd=ROOT)
    subprocess.run([sys.executable, "benchmarks/benchmark.py", "--device", "cuda",
                    "--dtype", "float16", "--rank", "16", "--tokens", "1,32",
                    "--output", "results/rank16.json"], check=True, cwd=ROOT)
    environment = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    (ROOT / "results/environment.txt").write_text(environment)


SKIPPED: required CUDA benchmarks and GPU environment capture.


## Measured Results

Rows are read only from this run's raw JSON. Speedup uses CUDA-event stream median.


In [10]:
if not HAS_CUDA:
    print("No GPU measurements available; no GPU speedup claimed.")
else:
    for filename in ("rank8.json", "rank16.json"):
        report = json.loads((ROOT / "results" / filename).read_text())
        rank = report["spec"]["rank"]
        for case in report["cases"]:
            methods = case["methods"]
            optimized = methods["factorized_optimized"]["cuda_event_stream_median_ms"]
            for name, result in methods.items():
                row = {"rank": rank, "tokens": case["tokens"], "method": name,
                       "cuda_event_stream_median_ms": result["cuda_event_stream_median_ms"],
                       "host_synchronized_median_ms": result["host_synchronized_median_ms"],
                       "preparation_ms": result["preparation_ms"],
                       "first_call_ms": result["first_call_ms"],
                       "memory": result["memory"], "correctness": result["correctness"]}
                if name == "factorized_optimized":
                    row["speedup_over_reference"] = (
                        methods["factorized_reference"]["cuda_event_stream_median_ms"] / optimized)
                    row["speedup_over_dense"] = (
                        methods["dense"]["cuda_event_stream_median_ms"] / optimized)
                print(json.dumps(row))


No GPU measurements available; no GPU speedup claimed.


## Limits and Reproduction

The kernel uses ordinary CUDA arithmetic, not additional precision or Tensor
Core optimization. At token 1 there are only Q=10 blocks and throughput may be
unfavorable. Shared-memory size grows with rank/modes; shapes exceeding the
device's ordinary per-block budget are rejected. No high-level CUDA contraction
fallback is used. CPU fallback supports the same algebra in PyTorch.

Run the stock suite with:

    PYTHONPATH=src python -m pytest -q

Run the two required benchmark commands above on a CUDA host. Open this notebook
in the same interpreter environment and Restart & Run All to regenerate the
modules and save GPU checks, JSON results, and profiler evidence.

For full Jupyter execution on a CUDA host, install notebook-only tools and run:

    python -m pip install jupyterlab ipykernel nbconvert
    python -m jupyter nbconvert --execute --to notebook --inplace solution.ipynb --ExecutePreprocessor.timeout=-1

Only solution.ipynb, src/factorized_inference/submission.py, and
src/factorized_inference/tr_kernels.py are implementation artifacts.
benchmarks/, tests/, reference.py, __init__.py, and pyproject.toml stay unchanged.
